# Scaled Dot-Product Attention: Desglose de la Ecuación y Conceptos Clave

A continuación repasamos la formulación matemática de **Scaled Dot-Product Attention** (Atención Producto Punto Escalado) del paper *Attention Is All You Need*, sus matrices y su intuición geométrica:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V$$

---

### 1. Componentes de la Ecuación

* **$A$ (Activaciones de Atención / Salida):** El resultado de la ecuación. Puedes pensar en estas activaciones como **ajustes** (pequeñas rotaciones o escalados en el espacio vectorial) que se combinan con otros tokens y se suman de vuelta a los embeddings de tokens originales (a través de la conexión residual).
* **$\text{softmax}$ ($\sigma$):** Función que normaliza las puntuaciones para convertirlas en una distribución de probabilidad válida (positivas y sumando $1.0$).
* **$Q$ (*Query* / Consulta):** Matriz que representa qué está buscando o pidiendo el token actual.
* **$K$ (*Key* / Clave):** Matriz que representa las etiquetas o el perfil de características de todos los tokens disponibles.
* **$\sqrt{d_k}$:** Factor de escala basado en la dimensionalidad de las claves ($d_k$).
* **$M$ (Máscara Causal):** Matriz triangular superior con $-\infty$ (opcional en encoders como BERT, obligatoria en decoders autorregresivos como GPT).
* **$V$ (*Value* / Valor):** Matriz con la información o contenido real que cada token transmitirá si hay coincidencia.
* **$W_O$ (Matriz de Proyección de Salida):** Matriz de pesos que transforma linealmente el resultado combinado de las cabezas de atención antes de proyectarlo a la siguiente capa.

---

### 2. El Factor de Escala ($\sqrt{d_k}$) y Dimensiones

Al multiplicar $Q \cdot K^T$, la varianza del producto escalar crece proporcionalmente a la dimensión de los vectores. Para evitar que las cifras crezcan demasiado y empujen a la función Softmax a regiones con gradientes casi nulos (desvanecimiento del gradiente o *vanishing gradients*), se divide entre $\sqrt{d_k}$:

* En **Atención de una sola cabeza (*Single-Head Attention*)**, la dimensión $d_k$ coincide con la dimensión de los embeddings ($d_{\text{model}} = 768$ en GPT-2 Small).
* En **Atención multicabezal (*Multi-Head Attention*)**, la dimensión total se divide entre el número de cabezas:
  $$d_k = \frac{d_{\text{model}}}{\text{num\_heads}} = \frac{768}{12} = 64$$

---

### 3. La Analogía: Una «App de Citas» para Tokens

*(Toda analogía tiene sus límites, pero ayuda a fijar la intuición)*

1. **El Perfil de Búsqueda ($Q$):** El vector $Q$ de un token actual es su perfil de citas; define qué tipo de información está buscando en los demás tokens.
2. **Los Perfiles Públicos ($K$):** Los vectores $K_1, K_2, K_3, \dots$ son los perfiles de los otros tokens en la secuencia.
3. **El Producto Punto ($Q \cdot K^T$):** Funciona como un cálculo de compatibilidad (similar al numerador de la similitud del coseno sin normalizar). Si $Q$ y $K$ apuntan en direcciones similares, el resultado es positivo y alto; si son incompatibles, da valores negativos.
4. **Softmax:** Toma estas puntuaciones brutas, amplifica las mejores coincidencias, atenúa las más bajas y las transforma en una **probabilidad de afinidad** que suma $1.0$.
5. **El Contenido Real ($V$):** Una vez definida la compatibilidad, el modelo extrae la información real almacenada en $V$ ponderada por dicha probabilidad.
6. **La Máscara Causal ($M$):** Garantiza que un token solo pueda hacer «match» con tokens de su pasado o presente, nunca del futuro.

---

### 4. ¿De Dónde Vienen $Q$, $K$ y $V$? (Pesos Estáticos vs. Activaciones Dinámicas)

Las matrices $Q$, $K$ y $V$ provienen de multiplicar la entrada $X$ (los embeddings de tokens actuales) por matrices de pesos entrenables:

$$Q = X W_Q \qquad K = X W_K \qquad V = X W_V$$

* **Pesos Estáticos ($W_Q, W_K, W_V$):** Se inicializan con valores aleatorios y se optimizan durante el preentrenamiento mediante descenso de gradiente. Una vez entrenado el modelo, **estos números son fijos y no cambian**.
* **Activaciones Dinámicas ($Q, K, V$):** Son el resultado de la multiplicación matricial $X \cdot W$. **Cambian dinámicamente** con cada secuencia de texto que procesa el modelo.

```python
import torch.nn as nn

# Implementación conceptual de las proyecciones lineales en PyTorch
query_layer = nn.Linear(n_embd, n_embd, bias=False)  # W_Q estática

# En la pasada hacia adelante (forward pass):
# x = embeddings(tokens)              -> Forma: (batch, seq_len, n_embd)
# Q = query_layer(x)                  -> Q dinámica generada por X @ W_Q